In [1]:
# 1. Import Libraries
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 2. Load Dataset
df = pd.read_csv('cars24data.csv')

# 3. Feature Extraction & Engineering
df['Brand'] = df['Model Name'].apply(lambda x: x.split()[1] if len(x.split()) > 1 else 'Other')

X = df.drop(columns=['Price', 'Model Name'])
y = df['Price']

categorical_cols = ['Brand', 'Spare key', 'Transmission', 'Fuel type']
numerical_cols = ['Manufacturing_year', 'Engine capacity', 'KM driven', 'Ownership', 'Imperfections', 'Repainted Parts']

# 4. Pipeline Setup
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 5. Train-Test Split & Fitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

# 6. Model Evaluation
y_pred = model.predict(X_test)
print("R² Score:", r2_score(y_test, y_pred)) # ~0.83 R²
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

# 7. Save PKL Model
with open('car_price_model.pkl', 'wb') as f:
    pickle.dump(model, f)

R² Score: 0.831375836777281
MAE: 54034.60207612457
RMSE: 74953.17730580401
